# ASSIS — FOD Detection Training (Phase 2)

End-to-end pipeline: clone the repo, get the FOD-A dataset, build the small-object-weighted split, train YOLOv8, benchmark against FAA AC 150/5220-24-style criteria, and try the result on a few images.

**Before running:** Runtime → Change runtime type → GPU.

This notebook is a thin runner around the scripts in `src/` — it does not duplicate their logic, so anything fixed in `src/` is automatically picked up here on the next `git pull`.

In [ ]:
# 1. Get the repository
# If you're running this notebook from inside a clone already, skip this cell.
!git clone https://github.com/sundevilaviator/ASSIS-FOD-Detection.git
%cd ASSIS-FOD-Detection

In [ ]:
# 2. Install dependencies
!pip install -q -r requirements.txt

## 3. Kaggle credentials

Get an API token from your Kaggle account (Settings → Create New Token), which downloads `kaggle.json`. Upload it here, or set the two environment variables directly.

In [ ]:
import os
from pathlib import Path

# Option A — upload kaggle.json interactively:
try:
    from google.colab import files
    uploaded = files.upload()  # select your kaggle.json when prompted
    Path('~/.kaggle').expanduser().mkdir(exist_ok=True)
    for fname in uploaded:
        Path(f'~/.kaggle/{fname}').expanduser().write_bytes(uploaded[fname])
    os.chmod(Path('~/.kaggle/kaggle.json').expanduser(), 0o600)
except ImportError:
    print('Not running in Colab — set KAGGLE_USERNAME / KAGGLE_KEY env vars instead, or place ~/.kaggle/kaggle.json manually.')

In [ ]:
# 4. Download FOD-A and build the small-object-weighted split
!python src/data_prep.py --download \
  --dataset kilogrand/foreign-object-debris-in-airports-fod-a-dataset \
  --out data/fod-a

!python src/data_prep.py --build-split \
  --source data/fod-a \
  --out data/fod-a-split \
  --small-object-max-area-pct 0.5 \
  --oversample-factor 3

In [ ]:
# 5. Train
# yolov8n.pt is faster to iterate on; swap to yolov8s.pt (configs/fod.yaml default)
# once the pipeline is confirmed working end-to-end.
!python src/train.py \
  --config configs/fod.yaml \
  --data data/fod-a-split/data.yaml \
  --model yolov8n.pt \
  --epochs 50

In [ ]:
# 6. Benchmark against FAA AC 150/5220-24-style criteria
# Find the most recent run's best weights automatically.
import glob
weights = sorted(glob.glob('runs/detect/train*/weights/best.pt'))[-1]
print('Using weights:', weights)

!python src/benchmark_faa.py \
  --weights {weights} \
  --data data/fod-a-split/data.yaml \
  --out docs/benchmark_results

In [ ]:
# 7. Quick visual check on a handful of test images
from ultralytics import YOLO
from IPython.display import Image as IPImage, display
import glob

model = YOLO(weights)
sample_images = glob.glob('data/fod-a-split/test/images/*')[:5]
results = model.predict(source=sample_images, conf=0.35, save=True, project='runs/infer', name='colab_preview', exist_ok=True)
for p in glob.glob('runs/infer/colab_preview/*.jpg')[:5]:
    display(IPImage(filename=p))

## 8. Save results

Copy the trained weights and benchmark report to Google Drive (or download them directly) so this run's evidence survives the Colab session ending. Then record the run in `docs/RESEARCH_LOG.md` back in your local clone before committing — that dated log entry, plus these artifacts, is what turns a training run into a citable piece of progress.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, datetime
stamp = datetime.datetime.utcnow().strftime('%Y%m%dT%H%M%SZ')
dest = f'/content/drive/MyDrive/ASSIS-FOD-runs/{stamp}'
os_makedirs = __import__('os').makedirs
os_makedirs(dest, exist_ok=True)
shutil.copy(weights, dest)
shutil.copytree('docs/benchmark_results', f'{dest}/benchmark_results', dirs_exist_ok=True)
print('Saved to', dest)